# CINF104 - Proyecto 1: Predicción GRD Hospital El Pino
## 03 - Entrenamiento y Evaluación de Modelos

Comparamos tres enfoques:  
1. **Baseline**: Random Forest  
2. **Gradient Boosting**: XGBoost / LightGBM  
3. **Red Neuronal**: MLP (Keras)

**Métricas:** Accuracy, Macro-F1, Top-3 Accuracy (dado el desbalance y 230 clases).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, json, time
from pathlib import Path

PROC_DIR  = Path('../data/processed')
MODEL_DIR = Path('../models')
REP_DIR   = Path('../reports')
MODEL_DIR.mkdir(parents=True, exist_ok=True)
REP_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42

# Cargar datasets
X_train = np.load(PROC_DIR / 'X_train.npy')
X_val   = np.load(PROC_DIR / 'X_val.npy')
X_test  = np.load(PROC_DIR / 'X_test.npy')
y_train = np.load(PROC_DIR / 'y_train.npy')
y_val   = np.load(PROC_DIR / 'y_val.npy')
y_test  = np.load(PROC_DIR / 'y_test.npy')

le     = joblib.load(PROC_DIR / 'label_encoder.pkl')
meta   = json.load(open(PROC_DIR / 'metadata.json'))

N_CLASES  = meta['n_clases']
N_FEATURES = meta['n_features']

print(f'Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}')
print(f'Clases: {N_CLASES} | Features: {N_FEATURES}')

## Funciones de evaluación

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

def top_k_accuracy(y_true, y_proba, k=3):
    top_k = np.argsort(y_proba, axis=1)[:, -k:]
    correct = np.array([y_true[i] in top_k[i] for i in range(len(y_true))])
    return correct.mean()

def evaluar(nombre, y_true, y_pred, y_proba=None):
    acc   = accuracy_score(y_true, y_pred)
    f1m   = f1_score(y_true, y_pred, average='macro', zero_division=0)
    f1w   = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    top3  = top_k_accuracy(y_true, y_proba, k=3) if y_proba is not None else float('nan')
    print(f'[{nombre}]  Acc={acc:.4f}  Macro-F1={f1m:.4f}  Weighted-F1={f1w:.4f}  Top-3={top3:.4f}')
    return {'modelo': nombre, 'accuracy': acc, 'macro_f1': f1m, 'weighted_f1': f1w, 'top3_acc': top3}

resultados = []

## Modelo 1: Random Forest (Baseline)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

t0 = time.time()
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=25,
    min_samples_leaf=2,
    n_jobs=-1,
    random_state=RANDOM_SEED
)
rf.fit(X_train, y_train)
print(f'Entrenado en {time.time()-t0:.1f}s')

y_pred_rf_val  = rf.predict(X_val)
y_proba_rf_val = rf.predict_proba(X_val)
res_rf_val = evaluar('RF (val)',  y_val,  y_pred_rf_val, y_proba_rf_val)

y_pred_rf_test  = rf.predict(X_test)
y_proba_rf_test = rf.predict_proba(X_test)
res_rf_test = evaluar('RF (test)', y_test, y_pred_rf_test, y_proba_rf_test)

resultados.append(res_rf_val)
joblib.dump(rf, MODEL_DIR / 'random_forest.pkl')
print('✅ Modelo guardado')

## Modelo 2: LightGBM

In [ ]:
try:
    import lightgbm as lgb
    HAS_LGBM = True
except ImportError:
    HAS_LGBM = False
    print('LightGBM no disponible. Instalar con: pip install lightgbm')

if HAS_LGBM:
    t0 = time.time()
    lgbm = lgb.LGBMClassifier(
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=63,
        min_child_samples=5,
        n_jobs=-1,
        random_state=RANDOM_SEED,
        verbose=-1
    )
    lgbm.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(50, verbose=False)]
    )
    print(f'Entrenado en {time.time()-t0:.1f}s')
    
    y_pred_lgbm_val  = lgbm.predict(X_val)
    y_proba_lgbm_val = lgbm.predict_proba(X_val)
    res_lgbm = evaluar('LightGBM (val)', y_val, y_pred_lgbm_val, y_proba_lgbm_val)
    
    y_pred_lgbm_test  = lgbm.predict(X_test)
    y_proba_lgbm_test = lgbm.predict_proba(X_test)
    evaluar('LightGBM (test)', y_test, y_pred_lgbm_test, y_proba_lgbm_test)
    
    resultados.append(res_lgbm)
    joblib.dump(lgbm, MODEL_DIR / 'lightgbm.pkl')
    print('✅ Modelo guardado')

## Modelo 3: Red Neuronal MLP (Keras)

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(RANDOM_SEED)

def build_mlp(input_dim, n_classes):
    inp = keras.Input(shape=(input_dim,), name='features')
    x = layers.Dense(512, activation='relu')(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    out = layers.Dense(n_classes, activation='softmax', name='output')(x)
    return keras.Model(inp, out)

model = build_mlp(N_FEATURES, N_CLASES)
model.summary()

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=10,
        restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=5,
        min_lr=1e-5, verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        filepath=str(MODEL_DIR / 'mlp_grd_best.keras'),
        save_best_only=True, monitor='val_accuracy', verbose=0
    )
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=256,
    callbacks=callbacks,
    verbose=1
)

## Curvas de entrenamiento

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'],     label='Train', color='steelblue')
axes[0].plot(history.history['val_accuracy'], label='Val',   color='coral')
axes[0].set_title('Accuracy por Época')
axes[0].set_xlabel('Época'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history.history['loss'],     label='Train', color='steelblue')
axes[1].plot(history.history['val_loss'], label='Val',   color='coral')
axes[1].set_title('Loss por Época')
axes[1].set_xlabel('Época'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(REP_DIR / 'curvas_entrenamiento_mlp.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Evaluación del MLP
y_proba_mlp_val  = model.predict(X_val,  verbose=0)
y_pred_mlp_val   = np.argmax(y_proba_mlp_val,  axis=1)
res_mlp_val = evaluar('MLP (val)',  y_val,  y_pred_mlp_val,  y_proba_mlp_val)

y_proba_mlp_test = model.predict(X_test, verbose=0)
y_pred_mlp_test  = np.argmax(y_proba_mlp_test, axis=1)
evaluar('MLP (test)', y_test, y_pred_mlp_test, y_proba_mlp_test)

resultados.append(res_mlp_val)

## Comparación de modelos

In [ ]:
df_res = pd.DataFrame(resultados)
print('=== COMPARACIÓN DE MODELOS (set de validación) ===')
print(df_res.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(df_res))
w = 0.25
ax.bar(x - w,   df_res['accuracy'],    w, label='Accuracy',       color='steelblue')
ax.bar(x,        df_res['macro_f1'],    w, label='Macro-F1',       color='coral')
ax.bar(x + w,    df_res['top3_acc'],    w, label='Top-3 Accuracy', color='seagreen')
ax.set_xticks(x)
ax.set_xticklabels(df_res['modelo'], rotation=15)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score')
ax.set_title('Comparación de Modelos — Validation Set')
ax.legend(); ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(REP_DIR / 'comparacion_modelos.png', dpi=150, bbox_inches='tight')
plt.show()

## Evaluación final del mejor modelo en test

In [ ]:
# Seleccionar el mejor modelo según Macro-F1
mejor = df_res.loc[df_res['macro_f1'].idxmax(), 'modelo']
print(f'Mejor modelo según Macro-F1 en val: {mejor}')
print()

# Reporte completo del MLP en test
print('=== Reporte completo MLP (test set) ===')
print(classification_report(
    y_test, y_pred_mlp_test,
    target_names=le.classes_,
    zero_division=0,
    labels=np.arange(N_CLASES)
)[:3000], '...')

## Matriz de confusión (Top 15 clases)

In [ ]:
from sklearn.metrics import confusion_matrix

# Filtrar solo las 15 clases más frecuentes en test
top_classes = pd.Series(y_test).value_counts().head(15).index.tolist()
mask = np.isin(y_test, top_classes)

y_test_top  = y_test[mask]
y_pred_top  = y_pred_mlp_test[mask]

cm = confusion_matrix(y_test_top, y_pred_top, labels=top_classes)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

labels_top = le.inverse_transform(top_classes)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=labels_top, yticklabels=labels_top,
            ax=ax, cbar_kws={'label': 'Proporción'})
ax.set_xlabel('Predicho'); ax.set_ylabel('Real')
ax.set_title('Matriz de Confusión Normalizada — Top 15 GRDs (MLP)')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(REP_DIR / 'confusion_matrix_top15.png', dpi=150, bbox_inches='tight')
plt.show()

## Guardado del modelo final

In [ ]:
model.save(MODEL_DIR / 'mlp_grd_final.keras')

# Guardar resumen de resultados
df_res.to_csv(REP_DIR / 'resultados_modelos.csv', index=False)

print('✅ Modelo final guardado en:', MODEL_DIR / 'mlp_grd_final.keras')
print('✅ Resultados guardados en:', REP_DIR / 'resultados_modelos.csv')